<a href="https://colab.research.google.com/github/sangjkim930/Project-Based-Learning/blob/main/Week_03_Gemini_API/Week3_Gemini_API_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# Week 3 Exercise: Evidence-Based Research with Gemini API
# ============================================================

# Install the Google GenAI SDK.
# Colab may already have this package installed.

!pip install -q google-genai

In [2]:
# Import the libraries needed for this exercise.

from google import genai
from google.genai import types
from google.colab import userdata, files

In [3]:
# ============================================================
# Connect to the Gemini API
# ============================================================

# Load the API key securely from Colab Secrets.
# The secret must be saved with the name:
# GEMINI_API_KEY

api_key = userdata.get("Gemini_API_Key")

# Create the Gemini API client.
client = genai.Client(api_key=api_key)

print("Gemini API connection is ready.")

Gemini API connection is ready.


In [4]:
# ============================================================
# Select the Gemini model
# ============================================================

MODEL_NAME = "gemini-3.8-flash"

print("Model:", MODEL_NAME)

Model: gemini-3.8-flash


In [5]:
# ============================================================
# Upload the three Nepal reports to Google Colab
# ============================================================

print("Please select the THREE PDF files provided for this exercise.")

uploaded_files = files.upload()

print("\nFiles uploaded to Colab:")
for filename in uploaded_files.keys():
    print("-", filename)

Please select the THREE PDF files provided for this exercise.


Saving 01_Nepal_Economy_2024_25_NRB.pdf to 01_Nepal_Economy_2024_25_NRB.pdf
Saving 02_Nepal_Development_Strategy_2025_29_ADB.pdf to 02_Nepal_Development_Strategy_2025_29_ADB.pdf
Saving 03_Nepal_Development_Update_2026_WorldBank.pdf to 03_Nepal_Development_Update_2026_WorldBank.pdf

Files uploaded to Colab:
- 01_Nepal_Economy_2024_25_NRB.pdf
- 02_Nepal_Development_Strategy_2025_29_ADB.pdf
- 03_Nepal_Development_Update_2026_WorldBank.pdf


In [7]:
# ============================================================
# Check whether the correct files were uploaded
# ============================================================

pdf_files = [
    "01_Nepal_Economy_2024_25_NRB.pdf",
    "02_Nepal_Development_Strategy_2025_29_ADB.pdf",
    "03_Nepal_Development_Update_2026_WorldBank.pdf"
]

missing_files = [
    filename for filename in pdf_files
    if filename not in uploaded_files
]

if len(missing_files) == 0:
    print("All three required files are ready.")
else:
    print("The following files are missing:")
    for filename in missing_files:
        print("-", filename)

All three required files are ready.


In [8]:
# ============================================================
# Upload the PDF files to Gemini
# ============================================================

gemini_docs = []

print("Uploading documents to Gemini...\n")

for file_path in pdf_files:

    uploaded = client.files.upload(file=file_path)

    gemini_docs.append(uploaded)

    print("Uploaded:", uploaded.display_name)

print("\nAll documents have been uploaded to Gemini.")

Uploading documents to Gemini...

Uploaded: None
Uploaded: None
Uploaded: None

All documents have been uploaded to Gemini.


In [9]:
# ============================================================
# Define evidence-based research instructions
# ============================================================

system_instruction = """
You are an evidence-based research assistant.

Use ONLY the three documents provided in this exercise.

For every important factual claim:

1. Identify the source document.
2. Provide the relevant section when it can be identified.
3. Provide a page number only when it can be clearly verified.
4. Do not invent page numbers, quotations, or citations.
5. If the documents do not provide enough evidence,
   clearly state that the evidence is not available.

Clearly distinguish information from different reports.
"""

In [12]:
# ============================================================
# Create a reusable Gemini research function
# with automatic retry
# ============================================================

import time
import random

def ask_gemini(question, max_retries=5):

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,

                contents=[
                    "SOURCE 1: Nepal Rastra Bank Annual Report 2024/25",
                    gemini_docs[0],

                    "SOURCE 2: ADB Nepal Country Partnership Strategy 2025-2029",
                    gemini_docs[1],

                    "SOURCE 3: World Bank Nepal Development Update 2026",
                    gemini_docs[2],

                    question
                ],

                config=types.GenerateContentConfig(
                    system_instruction=system_instruction
                )
            )

            return response.text

        except Exception as e:

            # Wait longer after each failed attempt.
            wait_time = (2 ** attempt) + random.random()

            print(f"Request failed. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)

    return "The request could not be completed after several attempts. Please try again later."

In [13]:
# ============================================================
# Query 1: Major development challenges
# ============================================================

question_1 = """
What are the three most important economic and development
challenges facing Nepal?

Support each challenge with evidence from the provided documents.

Present your answer in a table with the following columns:

Challenge | Explanation | Evidence | Source | Page or Section
"""

answer_1 = ask_gemini(question_1)

print(answer_1)

Based on the provided documents, the three most important economic and development challenges facing Nepal are:

1. **Low Economic Competitiveness, Underdeveloped Industry, and Heavy Dependence on Remittances**
2. **High Vulnerability to Climate Change, Natural Disasters, and Large Infrastructure Deficits**
3. **Sluggish Domestic Job Creation, Skills Mismatches, and High Youth Outmigration**

The table below details each challenge, accompanied by verified explanations and evidence from the reports:

| Challenge | Explanation | Evidence | Source | Page or Section |
| :--- | :--- | :--- | :--- | :--- |
| **1. Low Competitiveness, Underdeveloped Industry, and Heavy Remittance Dependence** | Nepal’s landlocked, mountainous geography inflates production and trade costs, causing structural transformation to bypass the industrial/manufacturing sector. The growth model relies heavily on consumption fueled by worker remittances, which contributes to "Dutch disease"-like effects (real effective 

In [14]:
# ============================================================
# Query 2: Development opportunities
# ============================================================

question_2 = """
What are the three most promising opportunities
for Nepal's future economic and social development?

Use only the provided documents.

For each opportunity, explain why it is important
and identify the supporting source.

Present the results in a clear table.
"""

answer_2 = ask_gemini(question_2)

print(answer_2)

Based on the provided documents, here are three of the most promising opportunities for Nepal’s future economic and social development:

| Promising Opportunity | Why It Is Important | Supporting Sources |
| :--- | :--- | :--- |
| **1. Hydropower Expansion and Cross-Border Clean Energy Trade** | • **Cornerstone of Economic Growth:** Hydropower provides clean, reliable energy, generates export revenue and foreign exchange, boosts productivity in energy-intensive industries (manufacturing, agro-processing), and attracts foreign investment.<br>• **Regional Energy Trade:** Nepal has seasonal surpluses during the wet season (June–November) that enable cross-border power trade. India committed to buying 10,000 MW of power over 10 years and approved an additional 280 MW of imports (totaling 1,216.7 MW), while Nepal began exporting 40 MW to Bangladesh.<br>• **Industrial Resilience:** Nearly 4,000 MW of projects licensed over recent years are under construction, underpinning industrial activity

In [15]:
# ============================================================
# Query 3: Compare perspectives
# ============================================================

question_3 = """
How do the Nepal Rastra Bank, Asian Development Bank,
and World Bank reports differ in their perspectives
on Nepal's economy and development?

For each report, identify:

1. Its main focus
2. Major issues emphasized
3. Important development priorities

Present the comparison in a table.
"""

answer_3 = ask_gemini(question_3)

print(answer_3)

Request failed. Retrying in 1.6 seconds...
Based on the provided documents, the **Nepal Rastra Bank (NRB)**, **Asian Development Bank (ADB)**, and **World Bank (WB)** examine Nepal’s economy and development from distinct perspectives:

* **Nepal Rastra Bank (NRB)** takes a **central banking, macroeconomic, and regulatory perspective**, focusing primarily on official statistics for FY 2024/25, monetary policy operations, price stability, external sector reserve management, financial sector regulation, and institutional operations.
* **Asian Development Bank (ADB)** adopts a **medium-term strategic development cooperation perspective** (Country Partnership Strategy 2025–2029), focusing on addressing structural obstacles, building institutional capacity for federalism, promoting private-sector-led growth and youth employability, and strengthening climate resilience.
* **World Bank (WB)** provides a **near-term macroeconomic monitoring and shock-impact perspective** (Nepal Development Upda

In [ ]:
# ============================================================
# YOUR TURN
# ============================================================

# Change the question below.
#
# Possible topics:
# - Infrastructure
# - Private-sector development
# - Youth employment
# - Digital technology
# - Tourism
# - Remittances
# - Foreign investment
#
# Write your own research question.

my_question = """
What role could digital technology play
in Nepal's future economic development?

Use evidence from the provided documents.
"""

my_answer = ask_gemini(my_question)

print(my_answer)